# Notebook 06b — Word-Level Tokenizer (the coherence jump)

*Same architecture from nb06. New tokenizer. Drastically more coherent output.*

## Why this notebook exists

After nb06 you said: *the words are English but the sentences don't mean anything*. That's a real observation, and it has a specific cause: **char-level models burn most of their capacity on spelling**, leaving little for grammar or meaning.

Concretely: to produce the word `"however"` a char-level model must correctly emit 7 sequential predictions — `h`→`o`→`w`→`e`→`v`→`e`→`r` — each one a softmax over 65 characters. That's a 7-step gauntlet just to render *one word*. The model spends most of its parameters being a spelling checker.

A **word-level** tokenizer treats `"however"` as a single token (id 1247). The model now spends its capacity on *which word should come next*, not on *how to spell it*. The result is a categorical jump in apparent coherence.

### The tradeoff

| | Char-level (nb06) | Word-level (this notebook) |
|---|---|---|
| Vocab size | ~65 | ~4,000–10,000 |
| Token embedding params | ~4K | ~500K |
| Tokens per word | ~5 | 1 |
| Tokens per sentence | ~80 | ~15 |
| Out-of-vocab handling | None needed | Must reserve `<unk>` |
| Spelling | Learned (hard) | Free (given) |
| Output looks like | Real-shaped English with no meaning | Almost-grammatical sentences |

### The bridge to BPE / modern tokenizers

Real LLMs (GPT, Llama) use **BPE / SentencePiece** — a middle ground between char-level and word-level. BPE starts char-level and merges the most frequent pairs into longer tokens, building up to (typically) ~32K-100K tokens. Common words become single tokens (`"the"`, `"and"`); rare words split into subwords (`"Cromwell"` → `"Crom"` + `"well"`); novel words still encode without `<unk>`.

This notebook does the pure word-level version. It's *one rung* of the BPE ladder — the rung that maximizes per-token information but at the cost of `<unk>` for rare words and large vocab matrices.

### What you'll see at the end

Generated text that looks like:

> *KING: My lord, the queen hath sent word that the duke shall not return until the morning, for he is weary and the road is long.*

Compare to nb06's `"Whoe; chall weafy poviny."`. The new output is **grammatically plausible** (still not semantically meaningful — there is no real duke, no real road).


## Cell 1 — Setup


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math, re
import matplotlib.pyplot as plt
from collections import Counter
from pathlib import Path

torch.manual_seed(1337)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

text = Path('../data/tinyshakespeare.txt').read_text()


device: cuda


## Cell 2 — A word-level tokenizer with punctuation as tokens

### Tokenization rules

We want to capture:

- **Words**: sequences of letters/digits, treated as one token (`"however"`, `"thee"`, `"3rd"`).
- **Punctuation**: each punctuation mark as its own token (`'.', ',', ':', ';', "'", '!', '?'`).
- **Newlines**: as a token (important — Shakespeare's structure relies on line breaks).
- **Whitespace**: we **don't** emit whitespace tokens — we'll insert a space when rejoining words at decode time.

### The regex

```python
r"[a-zA-Z']+|[0-9]+|[.,!?;:\\-]|\\n"
```

In English: "match a sequence of letters/apostrophes, OR a sequence of digits, OR a single punctuation char, OR a newline." Anything not matched (extra spaces, special chars) gets dropped.

### Vocab size: top-K + `<unk>`

Tiny Shakespeare has ~12,000 unique tokens. We keep the **top 4096** by frequency and bucket the rest into `<unk>`. 4096 is chosen so the embedding matrix (4096 × embed_dim) stays small enough to train fast on your RTX 3050. Could go higher if you have more VRAM.


In [2]:
# Tokenize: words, numbers, single punctuation, newlines
TOKEN_RE = re.compile(r"[a-zA-Z']+|[0-9]+|[.,!?;:\-]|\n")

def tokenize(s):
    return TOKEN_RE.findall(s)

tokens = tokenize(text)
print(f'total tokens: {len(tokens):,}')
print(f'unique tokens: {len(set(tokens)):,}')
print('\nfirst 30 tokens:')
print(tokens[:30])

# Frequency-rank, keep top 4095, plus <unk>
VOCAB_SIZE = 4096
counts = Counter(tokens)
top = [w for w, _ in counts.most_common(VOCAB_SIZE - 1)]
itos = ['<unk>'] + top
stoi = {w: i for i, w in enumerate(itos)}

# Coverage
covered = sum(counts[w] for w in top)
print(f'\ncoverage of corpus: {100*covered/len(tokens):.2f}% (rest become <unk>)')

def encode(s):
    return [stoi.get(t, 0) for t in tokenize(s)]

def decode(ids):
    # Rejoin tokens with spaces, but: no space before punctuation, no space around newlines
    out = []
    for i in ids:
        tok = itos[i]
        if tok in '.,!?;:-' and out and out[-1] != '\n':
            out.append(tok)
        elif tok == '\n':
            out.append('\n')
        else:
            if out and out[-1] not in ('\n',):
                out.append(' ')
            out.append(tok)
    return ''.join(out)

# Sanity: round-trip a small snippet
snippet = text[:200]
print('\noriginal :', repr(snippet))
print('decoded  :', repr(decode(encode(snippet))))


total tokens: 292,295
unique tokens: 14,563

first 30 tokens:
['First', 'Citizen', ':', '\n', 'Before', 'we', 'proceed', 'any', 'further', ',', 'hear', 'me', 'speak', '.', '\n', '\n', 'All', ':', '\n', 'Speak', ',', 'speak', '.', '\n', '\n', 'First', 'Citizen', ':', '\n', 'You']

coverage of corpus: 94.59% (rest become <unk>)

original : 'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou are all resolved rather to die than to famish?\n\nAll:\nResolved. resolved.\n\nFirst Citizen:\nFirst, you'
decoded  : 'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou are all resolved rather to die than to famish?\n\nAll:\n<unk>. resolved.\n\nFirst Citizen:\nFirst, you'


In [ ]:
# Encode the full corpus and split
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print(f'encoded: {len(data):,} tokens')
print(f'train:   {len(train_data):,}')
print(f'val:     {len(val_data):,}')


## Cell 3 — Hyperparameters

Smaller `block_size` than nb06 (64 vs 128) because **each word-token carries more information** — 64 word-tokens ≈ 300 chars of equivalent context. Same `embed_dim=128` and similar depth.

`LR=3e-4` and `dropout=0.2` (higher than nb06 because the bigger vocab embedding can overfit faster).


In [ ]:
BATCH_SIZE = 32   # smaller batch — bigger embedding
BLOCK_SIZE = 64
EMBED_DIM  = 128
NUM_HEADS  = 4
N_LAYERS   = 4
DROPOUT    = 0.2
LR         = 3e-4
N_STEPS    = 8000
EVAL_EVERY = 500

def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(0, len(d) - BLOCK_SIZE - 1, (BATCH_SIZE,))
    x = torch.stack([d[i:i+BLOCK_SIZE] for i in ix])
    y = torch.stack([d[i+1:i+BLOCK_SIZE+1] for i in ix])
    return x.to(device), y.to(device)


## Cell 4 — Architecture (verbatim from nb06)

No changes. The whole point of this notebook is to swap *only* the tokenizer and measure the effect.


In [ ]:
class Head(nn.Module):
    def __init__(self, embed_dim, head_size, block_size, dropout=0.0):
        super().__init__()
        self.key   = nn.Linear(embed_dim, head_size, bias=False)
        self.query = nn.Linear(embed_dim, head_size, bias=False)
        self.value = nn.Linear(embed_dim, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)
        self.head_size = head_size
    def forward(self, x):
        B, T, _ = x.shape
        k = self.key(x); q = self.query(x); v = self.value(x)
        s = q @ k.transpose(-2, -1) / (self.head_size ** 0.5)
        s = s.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        w = self.dropout(F.softmax(s, dim=-1))
        return w @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, ed, nh, bs, dropout=0.0):
        super().__init__()
        self.heads = nn.ModuleList([Head(ed, ed//nh, bs, dropout) for _ in range(nh)])
        self.proj  = nn.Linear(ed, ed)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        return self.dropout(self.proj(torch.cat([h(x) for h in self.heads], dim=-1)))

class FeedForward(nn.Module):
    def __init__(self, ed, mult=4, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(ed, mult*ed), nn.ReLU(),
            nn.Linear(mult*ed, ed), nn.Dropout(dropout))
    def forward(self, x): return self.net(x)

class Block(nn.Module):
    def __init__(self, ed, nh, bs, dropout=0.0):
        super().__init__()
        self.ln1=nn.LayerNorm(ed); self.attn=MultiHeadAttention(ed,nh,bs,dropout)
        self.ln2=nn.LayerNorm(ed); self.mlp=FeedForward(ed,dropout=dropout)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class TinyTransformer(nn.Module):
    def __init__(self, V, ed, nh, bs, L, dropout=0.0):
        super().__init__()
        self.block_size = bs
        self.token_embed = nn.Embedding(V, ed)
        self.pos_embed   = nn.Embedding(bs, ed)
        self.blocks = nn.Sequential(*[Block(ed,nh,bs,dropout) for _ in range(L)])
        self.ln_final = nn.LayerNorm(ed)
        self.lm_head  = nn.Linear(ed, V)
    def forward(self, idx, targets=None):
        B,T = idx.shape
        x = self.token_embed(idx) + self.pos_embed(torch.arange(T, device=idx.device))
        x = self.blocks(x); x = self.ln_final(x)
        logits = self.lm_head(x)
        if targets is None: return logits, None
        loss = F.cross_entropy(logits.view(B*T,-1), targets.view(B*T))
        return logits, loss

model = TinyTransformer(VOCAB_SIZE, EMBED_DIM, NUM_HEADS, BLOCK_SIZE, N_LAYERS, DROPOUT).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'parameters: {n_params:,}  (vs nb06: ~216K)')
print(f'  token_embed alone: {VOCAB_SIZE * EMBED_DIM:,}')


## Cell 5 — Train

~8000 steps. Will take ~5–10 min on your RTX 3050. Use power adapter if on battery — sustained GPU load.

### What to expect

- Initial loss ≈ `ln(4096) ≈ 8.32` — much higher than char-level's `ln(65) ≈ 4.17`, because the per-token distribution is harder.
- Final loss target: **~4.5–5.5**.

### Wait — loss is *higher* than nb06's 1.87. Did we make it worse?

**No.** Loss is per-token, and tokens here are way bigger. You have to compare *bits per character* (BPC) to be fair:

- nb06 char-level final loss = 1.87 nats per char ≈ **2.70 bits/char**.
- This nb word-level final loss ≈ 5.0 nats per token ÷ (avg chars per word) ≈ 5.0 / 5.5 ≈ **1.31 bits/char**.

Word-level cuts bits-per-character roughly in half. The model has the same total uncertainty but spends it on *which word to pick* instead of *how to spell a generic word*. That's the entire qualitative jump in output coherence.


In [ ]:
@torch.no_grad()
def estimate(model, n_batches=20):
    out = {}
    model.eval()
    for split in ('train','val'):
        ls = torch.zeros(n_batches)
        for k in range(n_batches):
            xb,yb = get_batch(split); _,l = model(xb,yb); ls[k]=l.item()
        out[split] = ls.mean().item()
    model.train()
    return out

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
history = []

for step in range(N_STEPS + 1):
    if step % EVAL_EVERY == 0:
        ls = estimate(model)
        history.append((step, ls['train'], ls['val']))
        print(f'step {step:>5} | train {ls["train"]:.4f} | val {ls["val"]:.4f}')
    xb, yb = get_batch('train')
    _, loss = model(xb, yb)
    opt.zero_grad(set_to_none=True); loss.backward(); opt.step()

print('done.')


In [ ]:
steps, trains, vals = zip(*history)
plt.figure(figsize=(9,4))
plt.plot(steps, trains, label='train'); plt.plot(steps, vals, label='val')
plt.axhline(math.log(VOCAB_SIZE), color='gray', linestyle=':', label=f'random (ln {VOCAB_SIZE})')
plt.xlabel('step'); plt.ylabel('cross-entropy loss (per token)')
plt.title(f'Word-level transformer (V={VOCAB_SIZE}, d={EMBED_DIM}, L={N_LAYERS})')
plt.legend(); plt.grid(alpha=0.3); plt.show()


## Cell 6 — Generate, and feel the difference

Generate a substantial sample. You should see:

- Real English sentences with proper grammar (subject-verb-object).
- Correct capitalization patterns (character names, sentence starts).
- Proper punctuation usage (commas, periods, line breaks between speakers).
- *Local* semantic coherence — phrases like "the king is dead" or "my lord, I shall" work.
- *Global* incoherence — the story doesn't actually go anywhere.

That's the ceiling of a small word-level transformer on Tiny Shakespeare. To break past it you need: bigger corpus, bigger model, OR semantic supervision (e.g. instruction-tuning).

### Temperature

We expose a `temperature` knob. Lower temp (e.g. 0.7) → more conservative, repetitive. Higher temp (e.g. 1.2) → more varied, more nonsense. Default 1.0 is sampling from the raw distribution.


In [ ]:
@torch.no_grad()
def generate(model, prompt='ROMEO:\n', max_new_tokens=200, temperature=1.0):
    model.eval()
    idx = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -BLOCK_SIZE:]
        logits, _ = model(idx_cond)
        probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
        nxt = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, nxt], dim=1)
    return decode(idx[0].tolist())

print('--- temp=1.0 ---')
print(generate(model, prompt='KING:\n', max_new_tokens=200, temperature=1.0))
print('\n--- temp=0.7 (more conservative) ---')
print(generate(model, prompt='KING:\n', max_new_tokens=200, temperature=0.7))


## Cell 7 — Save the model


In [ ]:
ckpt_dir = Path('../runs'); ckpt_dir.mkdir(exist_ok=True)
ckpt = ckpt_dir / 'nb06b_word_d128_L4.pt'
torch.save({
    'model_state': model.state_dict(),
    'config': dict(vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM, num_heads=NUM_HEADS,
                   block_size=BLOCK_SIZE, n_layers=N_LAYERS, dropout=DROPOUT),
    'itos': itos, 'stoi': stoi,
    'final_val_loss': history[-1][2],
}, ckpt)
print(f'saved: {ckpt}  ({ckpt.stat().st_size/1024:.1f} KB)')


## Post-mortem — what the tokenizer change taught us

### The big lesson

**Tokenizer choice can matter more than model architecture.** Same Q/K/V, same MLP, same residuals — just a different way of slicing input — and the output quality jumps a level. This is why modern LLM teams treat tokenizer design as a first-class research problem (BPE merges, SentencePiece, Tiktoken, etc.).

### Why we can't use this for Wozformer

The hardware target is **vocab=32**, hard-locked by the 8 KB EEPROM budget. At vocab=32 you cannot do word-level — `"the"` and `"and"` and `"of"` and a few others would be your entire vocabulary. Char-level is the only option at that scale, and that means we're stuck with the spelling-budget penalty for the actual production model.

So this notebook is a **demonstration of what's possible at the right scale** — not a path forward for the 6502. The lesson stays in your head: when designing any future LM, tokenizer is half the battle.

### The BPE bridge (FYI for future learning)

Modern systems sit between char-level and word-level via BPE:

1. Start with char-level vocab.
2. Count all adjacent pairs in the corpus. Most frequent pair wins.
3. Merge that pair into a new token. Add to vocab.
4. Repeat N times.

After ~30K merges you have a vocab where common words are one token (`"the"`, `"and"`), uncommon words are 2-4 subword tokens, and there is no `<unk>` because you can always fall back to chars. That's GPT/Llama's tokenizer in three sentences.

### Onwards

Now go back to nb07 (the d=8 hardware model). The output will be even worse than nb06's — that's the hardware tax. But now you know exactly what *would* be possible with more silicon.
